In [ ]:
# ============================================
# Phase A — Dataset Generation (30-Node Setup)
# ============================================

# --------------------------
# Cell 1 — Imports & Setup
# --------------------------
import numpy as np
import networkx as nx
import pandas as pd
from pathlib import Path
from dataclasses import dataclass
from typing import Tuple, Dict, Literal

# Data directory
DATA_DIR = Path("./tm_dataset")
DATA_DIR.mkdir(parents=True, exist_ok=True)

# --------------------------
# Cell 2 — Topologies & Generators
# --------------------------
@dataclass
class TopologySpec:
    name: str
    G: nx.Graph
    capacities: Dict[tuple, float]
    seed: int = None


def rocketfuel_like(n_nodes: int = 30,
                    p_backbone: float = 0.1,
                    p_access: float = 0.05,
                    capacity_range: Tuple[float, float] = (6.0, 14.0)) -> TopologySpec:
    nb = max(8, n_nodes // 3)

    # connected backbone
    while True:
        seed = np.random.randint(1 << 30)
        backbone = nx.erdos_renyi_graph(nb, p_backbone, seed=seed)
        if nx.is_connected(backbone):
            print(f"[Rocketfuel-like] Using seed {seed} for backbone")
            break

    mapping = {i: i + 1 for i in range(nb)}
    backbone = nx.relabel_nodes(backbone, mapping)
    G = nx.Graph()
    G.update(backbone)

    # attach access nodes
    for v in range(nb + 1, n_nodes + 1):
        deg_targets = np.random.choice(range(1, nb + 1),
                                       size=np.random.randint(1, 4),
                                       replace=False)
        for t in deg_targets:
            G.add_edge(v, int(t))

    # sparse access-access edges
    access_nodes = list(range(nb + 1, n_nodes + 1))
    for _ in range(int(0.03 * len(access_nodes) * (len(access_nodes) - 1) / 2)):
        u, v = np.random.choice(access_nodes, size=2, replace=False)
        if not G.has_edge(int(u), int(v)) and np.random.rand() < p_access:
            G.add_edge(int(u), int(v))

    low, high = capacity_range
    caps = {tuple(sorted(e)): float(np.random.uniform(low, high)) for e in G.edges()}
    return TopologySpec(f"rf_{n_nodes}_synth", G, caps, seed=seed)


# ---------- Traffic matrix generation ----------
def zero_diag(M: np.ndarray) -> np.ndarray:
    M = M.copy()
    np.fill_diagonal(M, 0.0)
    return M


def gen_uniform(N: int, low: float = 0.5, high: float = 5.0, rng=None) -> np.ndarray:
    if rng is None: rng = np.random.default_rng()
    return zero_diag(rng.uniform(low, high, size=(N, N)))


def gen_exponential(N: int, scale: float = 4.0, rng=None) -> np.ndarray:
    if rng is None: rng = np.random.default_rng()
    return zero_diag(rng.exponential(scale, size=(N, N)))


def gen_gravity(N: int, mu: float = 0.0, sigma: float = 1.0,
                noise_low: float = 0.8, noise_high: float = 1.2, rng=None) -> np.ndarray:
    if rng is None: rng = np.random.default_rng()
    w = rng.lognormal(mean=mu, sigma=sigma, size=N)
    TM = np.outer(w, w) * rng.uniform(noise_low, noise_high, size=(N, N))
    return zero_diag(TM)


def total_capacity(G: nx.Graph, capacities: Dict[tuple, float]) -> float:
    return float(sum(capacities[tuple(sorted(e))] for e in G.edges()))


def scale_tm(TM: np.ndarray, G: nx.Graph, capacities: Dict[tuple, float],
             load_level: float = 0.7, kappa: float = 3.0) -> np.ndarray:
    C_tot = total_capacity(G, capacities)
    D_target = kappa * load_level * C_tot
    curr_sum = float(TM.sum())
    if curr_sum <= 1e-9:
        return TM
    noise = np.random.uniform(0.85, 1.15)  # break perfect normalization
    return TM * (D_target / curr_sum) * noise


GenName = Literal["uniform", "exponential", "gravity"]

def sample_tm(N: int, topo: TopologySpec,
              mix: Dict[GenName, float] = {"exponential": 0.6, "gravity": 0.3, "uniform": 0.1},
              load_levels: Tuple[float, ...] = (0.3, 0.5, 0.7, 0.9, 1.1),
              kappa_range: Tuple[float, float] = (2.0, 4.0)) -> tuple:
    rng = np.random.default_rng(None)  # fresh RNG each call
    gens = list(mix.keys())
    probs = np.array([mix[g] for g in gens])
    probs /= probs.sum()
    gname = str(rng.choice(gens, p=probs))

    if gname == "uniform":
        TM = gen_uniform(N, rng=rng)
    elif gname == "exponential":
        TM = gen_exponential(N, rng=rng)
    else:
        TM = gen_gravity(N, rng=rng)

    L = float(rng.choice(load_levels))
    kappa = float(rng.uniform(*kappa_range))
    TM_scaled = scale_tm(TM, topo.G, topo.capacities, L, kappa)
    return TM_scaled, gname, L, kappa


# --------------------------
# Cell 3 — Config (30-Node)
# --------------------------
TOPO = rocketfuel_like(n_nodes=30)
N = TOPO.G.number_of_nodes()
N_SAMPLES = 2000
mix_cfg = {"exponential": 0.6, "gravity": 0.3, "uniform": 0.1}
load_levels = (0.3, 0.5, 0.7, 0.9, 1.1)
kappa_range = (2.0, 4.0)

# --------------------------
# Cell 4 — Build & Save Dataset
# --------------------------
all_TMs = np.zeros((N_SAMPLES, N, N), dtype=float)
meta_rows = []

for i in range(N_SAMPLES):
    # add slight capacity jitter every 200 samples
    if i % 200 == 0:
        for e in TOPO.G.edges():
            TOPO.capacities[e] = np.random.uniform(6.0, 14.0)

    TM, gname, L, kappa = sample_tm(N, TOPO, mix=mix_cfg,
                                    load_levels=load_levels,
                                    kappa_range=kappa_range)
    all_TMs[i] = TM
    meta_rows.append((gname, L, kappa))

topo_name = f"{TOPO.name}_seed{TOPO.seed}"

tm_npy = DATA_DIR / f"{topo_name}_TMs.npy"
tm_csv = DATA_DIR / f"{topo_name}_TMs_meta.csv"
np.save(tm_npy, all_TMs)
pd.DataFrame(meta_rows, columns=["gen", "load_level", "kappa"]).to_csv(tm_csv, index=False)

edges_sorted = [tuple(sorted(e)) for e in TOPO.G.edges()]
caps_ordered = [TOPO.capacities[e] for e in edges_sorted]
topo_npz = DATA_DIR / f"{topo_name}_topology.npz"
np.savez(topo_npz,
         name=TOPO.name,
         nodes=np.array(list(TOPO.G.nodes()), dtype=int),
         edges=np.array(edges_sorted, dtype=int),
         capacities=np.array(caps_ordered, dtype=float))

print(f"[Saved Dataset] {tm_npy}, {tm_csv}")
print(f"[Saved Topology] {topo_npz}")

# --------------------------
# Cell 5 — Quick Preview
# --------------------------
TMS = np.load(tm_npy)
META = pd.read_csv(tm_csv)
print("Dataset shape:", TMS.shape)
print(META.head())

rows = []
for idx in np.random.choice(len(TMS), size=3, replace=False):
    rows.append({
        "idx": int(idx),
        "gen": META.loc[idx, "gen"],
        "load_level": float(META.loc[idx, "load_level"]),
        "kappa": float(META.loc[idx, "kappa"]),
        "sum_demand": float(TMS[idx].sum()),
        "max_demand": float(TMS[idx].max())
    })
print(pd.DataFrame(rows))

example_path = DATA_DIR / f"{TOPO.name}_example_TM.csv"
pd.DataFrame(TMS[int(rows[0]['idx'])]).to_csv(example_path, index=False)
print(f"[Saved Example TM] {example_path}")

npz = np.load(topo_npz, allow_pickle=True)
nodes = npz["nodes"].astype(int).tolist()
edges = [tuple(map(int, e)) for e in npz["edges"]]
capacities_vals = npz["capacities"].astype(float).tolist()
print(f"[Reloaded Topology] {npz['name']}")
print("Nodes:", len(nodes), "Edges:", len(edges))
print("Sample capacities:", capacities_vals[:5])


In [ ]:
import numpy as np

TMS = np.load("tm_dataset/rf_30_synth_seed271716155_TMs.npy")

TMS_flat = TMS.reshape(len(TMS), -1)
corr_matrix = np.corrcoef(TMS_flat)
upper_idx = np.triu_indices_from(corr_matrix, k=1)
avg_corr = np.mean(np.abs(corr_matrix[upper_idx]))

print(f"Average inter-TM correlation: {avg_corr:.3f}")


In [ ]:
tm_sums = TMS.reshape(len(TMS), -1).sum(axis=1)
print(f"TM sum stats — min: {tm_sums.min():.2f}, max: {tm_sums.max():.2f}, std: {tm_sums.std():.2f}")


In [ ]:
# =======================================
# Phase B — RL TRAINING & TRANSFER TESTS
# =======================================

# Cell 0 — Load dataset, split, rebuild graph state, and precompute/load paths

import numpy as np, pandas as pd, networkx as nx, pickle
from pathlib import Path
from networkx.algorithms.simple_paths import shortest_simple_paths

DATA_DIR = Path("./tm_dataset")
TOPO_NAME = "rf_30_synth_seed271716155"

# --- Load traffic matrices + metadata ---
TMS = np.load(DATA_DIR / f"{TOPO_NAME}_TMs.npy")
META = pd.read_csv(DATA_DIR / f"{TOPO_NAME}_TMs_meta.csv")

# --- Load topology ---
topo_npz = np.load(DATA_DIR / f"{TOPO_NAME}_topology.npz", allow_pickle=True)
nodes = topo_npz["nodes"].astype(int).tolist()
edges_arr = topo_npz["edges"]
edges = [tuple(map(int, e)) for e in edges_arr]
capacities_vals = topo_npz["capacities"].tolist()

# Build graph + capacities
G = nx.Graph(); G.add_nodes_from(nodes); G.add_edges_from(edges)
cap_dict = {tuple(sorted(e)): float(c) for e, c in zip(edges, capacities_vals)}
EDGE_LIST = [tuple(sorted(e)) for e in G.edges()]
EDGE_INDEX = {e:i for i,e in enumerate(EDGE_LIST)}

# --- Train/test split ---
n_total = len(TMS); n_train = int(0.7 * n_total)
TMS_train, META_train = TMS[:n_train], META.iloc[:n_train].reset_index(drop=True)
TMS_test,  META_test  = TMS[n_train:],  META.iloc[n_train:].reset_index(drop=True)

# --- Candidate paths (with caching + parallel + progress) ---
# --- Candidate paths (FAST: only 1 shortest path per pair) ---
from tqdm import tqdm
import pickle
import networkx as nx

k_paths = 3
paths_file = DATA_DIR / f"{TOPO_NAME}_candidate_paths_fast.pkl"

def compute_single_shortest(G, src, dst):
    try:
        path = nx.shortest_path(G, src, dst)  # built-in BFS / Dijkstra
    except nx.NetworkXNoPath:
        path = []  # handle disconnected pairs
    return path

if paths_file.exists():
    print(f"[Info] Loading candidate paths from {paths_file}")
    with open(paths_file, "rb") as f:
        candidate_paths = pickle.load(f)
else:
    print("[Info] Computing single shortest paths quickly...")
    candidate_paths = {}
    for s in tqdm(nodes, desc="Src"):
        for d in nodes:
            if s != d:
                p = compute_single_shortest(G, s, d)
                if p:
                    candidate_paths[(s, d)] = [p]  # wrap in list for consistency

    with open(paths_file, "wb") as f:
        pickle.dump(candidate_paths, f)
    print(f"[Info] Saved candidate paths to {paths_file}")

print(f"Candidate paths ready for {len(candidate_paths)} src-dst pairs "
      f"(1 path each).")

import random

# --- Verification: sample a few random pairs ---
sample_pairs = random.sample(list(candidate_paths.keys()), 5)
print("\n[Verification] Sample candidate paths:")
for (s, d) in sample_pairs:
    paths = candidate_paths[(s, d)]
    print(f"  Pair ({s} → {d}):")
    for i, p in enumerate(paths, 1):
        print(f"    Path {i}: {p}")

# --- Coverage check ---
counts = [len(v) for v in candidate_paths.values()]
print("\n[Sanity] Path count distribution:")
print(f"  Min paths: {min(counts)}")
print(f"  Max paths: {max(counts)}")
print(f"  Pairs with <{k_paths} paths: {sum(c < k_paths for c in counts)} / {len(counts)}")


In [ ]:
# Cell 1 — Torch config & dimensions

import torch, torch.nn as nn, torch.optim as optim, random
from collections import deque

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

N = TMS.shape[1]
N_ACTIONS = N * (N - 1)                     # all ordered src->dst (dst!=src)
STATE_DIM = N*N + N_ACTIONS + len(EDGE_LIST)  # TM + mask + link loads

print(f"N={N}, Actions={N_ACTIONS}, Edges={len(EDGE_LIST)}, STATE_DIM={STATE_DIM}")



In [ ]:
# ==============================================
# Cell 2 — φ(s) encoder + DS-DQN (two-time-scale with φ(s)–φ(s′) decorrelation)
# ==============================================

import torch
import torch.nn as nn
import torch.optim as optim

# ----------------------------------------------
# Encoder: φ(s)
# ----------------------------------------------
class PhiEncoder(nn.Module):
    def __init__(self, state_dim, phi_dim=None):
        super().__init__()
        if phi_dim is None:
            phi_dim = state_dim   # match output dim to input by default

        self.fc1 = nn.Linear(state_dim, 256)
        self.fc2 = nn.Linear(256, 128)
        self.fc3 = nn.Linear(128, phi_dim)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        return self.fc3(x)  # φ(s)

# Debug check (optional)
phi_encoder = PhiEncoder(STATE_DIM, phi_dim=None).to(device)
dummy = torch.zeros(1, STATE_DIM).to(device)
out = phi_encoder(dummy)
print("Input dim:", dummy.shape, "Output dim:", out.shape)


# ----------------------------------------------
# DS-DQN with two-time-scale learning
# ----------------------------------------------
class DS_DQN(nn.Module):
    def __init__(self,
                 state_dim,
                 n_actions,
                 phi_dim=None,
                 lr_enc=3e-5,     # α (slow encoder)
                 lr_w=3e-3):      # β (fast head)
        super().__init__()
        if phi_dim is None:
            phi_dim = state_dim

        self.encoder = PhiEncoder(state_dim, phi_dim)
        self.w = nn.Parameter(torch.randn(n_actions, phi_dim) * 0.01)

        # Separate optimizers for two-time-scale learning
        self.opt_enc = optim.Adam(self.encoder.parameters(), lr=lr_enc)
        self.opt_w   = optim.Adam([self.w], lr=lr_w)

    # ------------------------------------------------
    # Forward pass
    # ------------------------------------------------
    def Q(self, s):
        """Compute Q-values and feature embeddings φ(s)."""
        phi = self.encoder(s)                 # [B, φ_dim]
        phi = nn.functional.normalize(phi, dim=1)
        q = torch.matmul(phi, self.w.T)       # [B, n_actions]
        return q, phi

    # ------------------------------------------------
    # Correlation penalty: L2(φ)
    # ------------------------------------------------
    def _corr_penalty_random_pairs(self, phi_s):
        """
        Compute decorrelation loss between random pairs of φ(s)
        drawn from the same batch.
        """
        batch_size = phi_s.size(0)
        perm = torch.randperm(batch_size)
        phi_s_perm = phi_s[perm]

        # Compute per-sample Pearson correlation between φ(s) and φ(s_perm)
        phi_s_centered = phi_s - phi_s.mean(dim=1, keepdim=True)
        phi_s_perm_centered = phi_s_perm - phi_s_perm.mean(dim=1, keepdim=True)

        numerator = torch.sum(phi_s_centered * phi_s_perm_centered, dim=1)
        denominator = torch.sqrt(
            torch.sum(phi_s_centered**2, dim=1) * torch.sum(phi_s_perm_centered**2, dim=1)
        ) + 1e-8

        corr = numerator / denominator
        return (corr ** 2).mean()  # penalize squared correlation


    # ------------------------------------------------
    # Update step
    # ------------------------------------------------
    def update(self, batch, target_net, gamma=0.99, lambda_reg=0.0):
        """
        One DS-DQN update:
          - L1(φ,w): TD loss
          - L2(φ): decorrelation between random φ(s) pairs
          - Two-time-scale updates (w fast, θ₁ slow)
        """
        s, a, r, s_next, done = batch
        s      = torch.FloatTensor(s).to(device)
        s_next = torch.FloatTensor(s_next).to(device)
        a      = torch.LongTensor(a).to(device)
        r      = torch.FloatTensor(r).to(device)
        done   = torch.FloatTensor(done).to(device)

        # ----- Q(s,a) -----
        q_vals, phi_s = self.Q(s)
        q_sa = q_vals.gather(1, a.unsqueeze(1)).squeeze(1)

        # ----- Target network -----
        with torch.no_grad():
            q_next, _ = target_net.Q(s_next)
            y = r + gamma * (1 - done) * q_next.max(1)[0]

        # ----- Losses -----
        td_loss = (y - q_sa).pow(2).mean()               # value accuracy
        l2_loss = self._corr_penalty_random_pairs(phi_s) # decorrelation term
        loss = td_loss + lambda_reg * l2_loss

        # ----- Backprop (two-time-scale) -----
        self.opt_enc.zero_grad()
        self.opt_w.zero_grad()
        loss.backward()
        self.opt_w.step()     # fast head (β)
        self.opt_enc.step()   # slow encoder (α)

        return td_loss.item(), l2_loss.item()


In [ ]:
# Cell 3 — Replay buffer

class ReplayBuffer:
    def __init__(self, capacity=10000):
        self.buffer = deque(maxlen=capacity)
    def push(self, s, a, r, s_next, done):
        self.buffer.append((s, a, r, s_next, done))
    def sample(self, batch_size=256):
        batch = random.sample(self.buffer, batch_size)
        s, a, r, s_next, done = map(np.array, zip(*batch))
        return s, a, r, s_next, done
    def __len__(self):
        return len(self.buffer)


In [ ]:
# Cell 4 — Environment helpers (state, action mapping, reward, with τ filtering)

def flow_index_to_pair(a: int, N: int) -> tuple:
    src = a // (N - 1)
    dst = a % (N - 1)
    if dst >= src: 
        dst += 1
    return src, dst  # 0-based (align with TM indexing)


def init_state(TM):
    mask = np.zeros(N_ACTIONS, dtype=float)
    loads = np.zeros(len(EDGE_LIST), dtype=float)
    return np.concatenate([TM.flatten(), mask, loads])


def apply_action(state, action, TM, lam=0.5, tau=0.8):
    """
    One environment step:
    - Pick action (src,dst)
    - Route demand using precomputed candidate paths[(src,dst)]
    - Apply τ-threshold filtering on link utilizations
    - Update loads and compute reward
    """
    TM_flat = state[:N*N]
    mask = state[N*N:N*N+N_ACTIONS]
    loads = state[N*N+N_ACTIONS:]

    TM_mat = TM_flat.reshape(N, N).copy()
    if mask[action] == 1.0:
        # Already routed this flow → small penalty
        U = max(loads[i] / cap_dict[EDGE_LIST[i]] for i in range(len(EDGE_LIST)))
        rho = np.mean([loads[i] / cap_dict[EDGE_LIST[i]] for i in range(len(EDGE_LIST))])
  #      r = (1-lam)*(1-U) - lam*rho - 0.1
        r = (1 - lam) * (1 - U) + lam * (1 - rho) - 0.1
        return state.copy(), r, U, rho

    # Mark this flow as routed
    mask_new = mask.copy()
    mask_new[action] = 1.0

    s, d = flow_index_to_pair(action, N)
    demand = TM_mat[s, d]

    # --- Path selection with τ filtering ---
    paths = candidate_paths[(s+1, d+1)]  # precomputed (1-based nodes in G)
    chosen_path = None

    for path in paths:  # check each candidate path
        safe = True
        for u, v in zip(path[:-1], path[1:]):
            e = tuple(sorted((u, v))); idx = EDGE_INDEX[e]
            projected_util = (loads[idx] + demand) / cap_dict[e]
            if projected_util > tau:
                safe = False
                break
        if safe:
            chosen_path = path
            break

    # If no safe path, fallback: pick first candidate anyway (with penalty)
    if chosen_path is None:
        chosen_path = paths[0]

    # --- Update link loads ---
    loads_new = loads.copy()
    for u, v in zip(chosen_path[:-1], chosen_path[1:]):
        e = tuple(sorted((u, v))); idx = EDGE_INDEX[e]
        loads_new[idx] += demand

    # --- Compute utilization & reward ---
    utilizations = np.array([loads_new[i] / cap_dict[EDGE_LIST[i]] for i in range(len(EDGE_LIST))])
    U = float(utilizations.max())
    rho = float(utilizations.mean())
  #  r = (1 - lam) * (1 - U) - lam * rho
    r = (1 - lam) * (1 - U) + lam * (1 - rho)


    s_next = np.concatenate([TM_mat.flatten(), mask_new, loads_new])
    return s_next, r, U, rho


In [ ]:
# Cell 4a — Quick debug check
state0 = init_state(TMS_train[0])
s_next, r, U, rho = apply_action(state0, action=0, TM=TMS_train[0], lam=0.55, tau=0.8)

print("STATE_DIM check:", STATE_DIM)
print("Initial state shape:", state0.shape)
print("Next state shape:", s_next.shape)
print("Reward example:", r, "U:", U, "rho:", rho)


In [ ]:
# Cell 5 — Training loop (DS-DQN with ε-decay + τ filtering paths)
import os
SAVE_DIR = "./plots"
os.makedirs(SAVE_DIR, exist_ok=True)

EPISODES = 3000          # longer training
K = 30                    # steps per episode
BATCH_SIZE = 256
GAMMA = 0.99
TARGET_UPDATE = 50

# ε-greedy schedule
eps_start = 1.0
eps_end = 0.05
eps_decay = 0.995
eps = eps_start

# DS-DQN networks
main_net = DS_DQN(STATE_DIM, N_ACTIONS, phi_dim=None,
                  lr_enc=1e-4, lr_w=3e-3).to(device)
target_net = DS_DQN(STATE_DIM, N_ACTIONS, phi_dim=None,
                    lr_enc=1e-4, lr_w=3e-3).to(device)
target_net.encoder.load_state_dict(main_net.encoder.state_dict())
target_net.w.data.copy_(main_net.w.data)

replay = ReplayBuffer(10000)

# Logs
rewards_log, U_log, rho_log = [], [], []
td_log, corr_log, gen_log = [], [], []
w_norms, enc_norms = [], []
corr_ep_log = []

# λ schedule
LAMBDA_MAX = 0.6
#DELTA_LAMBDA = LAMBDA_MAX / max(1, EPISODES / 2)
DELTA_LAMBDA = LAMBDA_MAX / 3000
lambda_reg = 0.0

for ep in range(EPISODES):
    idx = np.random.randint(len(TMS_train))
    TM = TMS_train[idx]
    tm_type = META_train.iloc[idx]["gen"]
    state = init_state(TM)

    ep_rewards, ep_Us, ep_rhos = [], [], []
    ep_td_losses, ep_corr_losses = [], []

    for step in range(K):
        # ε-greedy
        s_tensor = torch.FloatTensor(state).unsqueeze(0).to(device)
        q_vals, _ = main_net.Q(s_tensor)
        if random.random() < eps:
            action = np.random.randint(N_ACTIONS)   # explore
        else:
            action = q_vals.argmax(1).item()        # exploit

        # env transition (now uses τ-filtering paths)
        s_next, r, U, rho = apply_action(state, action, TM, lam=0.5, tau=0.8)
        replay.push(state, action, r, s_next, 0.0)
        state = s_next

        ep_rewards.append(r); ep_Us.append(U); ep_rhos.append(rho)

        # learn
        if len(replay) >= BATCH_SIZE:
            batch = replay.sample(BATCH_SIZE)
            td, corr = main_net.update(batch, target_net, gamma=GAMMA, lambda_reg=lambda_reg)
            td_log.append(td); corr_log.append(corr)
            ep_td_losses.append(td); ep_corr_losses.append(corr)
            
            # Track parameter norms
            w_norms.append(main_net.w.detach().norm().item())
            enc_norms.append(sum(p.detach().norm().item() for p in main_net.encoder.parameters()))

    # episode logs
    rewards_log.append(float(np.mean(ep_rewards)))
    U_log.append(float(np.mean(ep_Us)))
    rho_log.append(float(np.mean(ep_rhos)))
    gen_log.append(tm_type)
    

    # ---- summary ----
    td_ep = np.mean(ep_td_losses) if ep_td_losses else float("nan")
    corr_ep = np.mean(ep_corr_losses) if ep_corr_losses else float("nan")
    corr_ep_log.append(corr_ep) 
    print(
        f"[Ep {ep+1:04d}] "
        f"eps={eps:.3f} λ={lambda_reg:.4f} "
        f"R={np.mean(ep_rewards):+.4f} U={np.mean(ep_Us):.3f} ρ={np.mean(ep_rhos):.3f} "
        f"TD={td_ep:.5f} Corr={corr_ep:.5f} "
        f"replay={len(replay)}"
    )
    print(f"Episode {ep+1}, total routed flows: {K}, avg reward: {np.mean(ep_rewards):.4f}")
    
    # target net sync
    if (ep + 1) % TARGET_UPDATE == 0:
        target_net.encoder.load_state_dict(main_net.encoder.state_dict())
        target_net.w.data.copy_(main_net.w.data)

    # ramp λ
    lambda_reg = min(lambda_reg + DELTA_LAMBDA, LAMBDA_MAX)

    # decay ε
    eps = max(eps_end, eps * eps_decay)

# Save encoder φ(s)
phi_path = DATA_DIR / f"{TOPO_NAME}_phi_trained_DSDQN.pth"
torch.save(main_net.encoder.state_dict(), phi_path)
print(f"Saved encoder to {phi_path}")

# Save logs to CSV
df_summary = pd.DataFrame({
    "episode": np.arange(len(rewards_log)),
    "reward": rewards_log,
    "U": U_log,
    "rho": rho_log,
    "corr_loss": corr_ep_log
})
out_csv = f"{SAVE_DIR}/train_reward_utilization_corr_summary_dsdqn.csv"
df_summary.to_csv(out_csv, index=False)
print(f"Saved: {out_csv}")

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import os

SAVE_DIR = "./plots"
os.makedirs(SAVE_DIR, exist_ok=True)

# Load vanilla DQN logs
df = pd.read_csv(f"{SAVE_DIR}/train_reward_utilization_corr_summary_dsdqn.csv")
corr_ep_log = df["corr_loss"].values

# Rolling mean helper
def roll(x, w=50):
    return pd.Series(x).rolling(window=w, min_periods=1).mean().to_numpy()

ROLL_WIN = 50

plt.figure()
plt.plot(
    roll(corr_ep_log, ROLL_WIN),
    color="crimson",
    linewidth=2.3,
    label=f"{ROLL_WIN}-ep avg"
)

plt.title("Mean Correlation Loss per Episode (DS-DQN)",
          fontsize=16, weight='semibold')
plt.xlabel("Episode", fontsize=15, weight='bold')
plt.ylabel("Mean correlation penalty", fontsize=15, weight='bold')

plt.xticks(fontsize=13, weight='bold')
plt.yticks(fontsize=13, weight='bold')
plt.tick_params(axis='both', which='major', width=1.2, length=6)
plt.grid(True, linestyle="--", alpha=0.4)
plt.legend(fontsize=13, loc="upper right", frameon=False)

plt.tight_layout()
plt.savefig(f"{SAVE_DIR}/train_corrloss_ds-dqn.png",
            format='eps', dpi=600, bbox_inches="tight")
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import os

SAVE_DIR = "./plots"
os.makedirs(SAVE_DIR, exist_ok=True)

# Load both CSV logs
df_ds = pd.read_csv(f"{SAVE_DIR}/train_reward_utilization_corr_summary_dsdqn.csv")                 # DS-DQN
df_vanilla = pd.read_csv(f"{SAVE_DIR}/train_reward_utilization_corr_summary_vanilla.csv")    # Vanilla DQN

# Extract correlation logs
corr_ds = df_ds["corr_loss"].values
corr_vanilla = df_vanilla["corr_loss"].values

# Rolling mean helper
def roll(x, w=50):
    return pd.Series(x).rolling(window=w, min_periods=1).mean().to_numpy()

ROLL_WIN = 50

# ==========================================================
# Combined Correlation Plot: DS-DQN vs Vanilla DQN
# ==========================================================
plt.figure()

plt.plot(
    roll(corr_ds, ROLL_WIN),
    color="darkorange", linewidth=2.3,
    label="DS-DQN"
)
plt.plot(
    roll(corr_vanilla, ROLL_WIN),
    color="crimson", linewidth=2.3,
    label="Vanilla DQN"
)

# Titles and labels
plt.title("Mean Correlation Loss per Episode", fontsize=16, weight='semibold')
plt.xlabel("Episode", fontsize=15, weight='bold')
plt.ylabel("Mean correlation penalty", fontsize=15, weight='bold')

# Axis styling
plt.xticks(fontsize=13, weight='bold')
plt.yticks(fontsize=13, weight='bold')
plt.tick_params(axis='both', which='major', width=1.2, length=6)

# Grid and legend
plt.grid(True, linestyle="--", alpha=0.35)
plt.legend(fontsize=13, loc="upper right", frameon=False)

# Save high-quality EPS for Overleaf
plt.tight_layout()
plt.savefig(f"{SAVE_DIR}/train_corrloss_comparison.eps",
            format='eps', dpi=600, bbox_inches="tight")
plt.show()


In [ ]:
# === Cell 5 — Training loop (Vanilla DQN, no decorrelation regularization) ===

EPISODES = 3000
K = 30
BATCH_SIZE = 256
GAMMA = 0.99
TARGET_UPDATE = 50

# ε-greedy schedule
eps_start = 1.0
eps_end = 0.05
eps_decay = 0.995
eps = eps_start

# Vanilla DQN (same DS-DQN architecture, but λ=0 disables decorrelation)
main_net = DS_DQN(STATE_DIM, N_ACTIONS, phi_dim=None,
                  lr_enc=1e-4, lr_w=3e-3).to(device)
target_net = DS_DQN(STATE_DIM, N_ACTIONS, phi_dim=None,
                    lr_enc=1e-4, lr_w=3e-3).to(device)
target_net.encoder.load_state_dict(main_net.encoder.state_dict())
target_net.w.data.copy_(main_net.w.data)

replay = ReplayBuffer(10000)

# Logs
rewards_log, U_log, rho_log = [], [], []
td_log, corr_log, gen_log = [], [], []
w_norms, enc_norms = [], []
corr_ep_log = []

# No correlation regularization
lambda_reg = 0.0

print("\n=== Running Vanilla DQN (λ=0, no decorrelation) ===")

for ep in range(EPISODES):
    idx = np.random.randint(len(TMS_train))
    TM = TMS_train[idx]
    tm_type = META_train.iloc[idx]["gen"]
    state = init_state(TM)

    ep_rewards, ep_Us, ep_rhos = [], [], []
    ep_td_losses, ep_corr_losses = [], []

    for step in range(K):
        s_tensor = torch.FloatTensor(state).unsqueeze(0).to(device)
        q_vals, _ = main_net.Q(s_tensor)
        if random.random() < eps:
            action = np.random.randint(N_ACTIONS)
        else:
            action = q_vals.argmax(1).item()

        s_next, r, U, rho = apply_action(state, action, TM, lam=0.5, tau=0.8)
        replay.push(state, action, r, s_next, 0.0)
        state = s_next

        ep_rewards.append(r); ep_Us.append(U); ep_rhos.append(rho)

        if len(replay) >= BATCH_SIZE:
            batch = replay.sample(BATCH_SIZE)
            td, corr = main_net.update(batch, target_net, gamma=GAMMA, lambda_reg=lambda_reg)
            td_log.append(td); corr_log.append(corr)
            ep_td_losses.append(td); ep_corr_losses.append(corr)

    rewards_log.append(np.mean(ep_rewards))
    U_log.append(np.mean(ep_Us))
    rho_log.append(np.mean(ep_rhos))
    gen_log.append(tm_type)

    td_ep = np.mean(ep_td_losses) if ep_td_losses else float("nan")
    corr_ep = np.mean(ep_corr_losses) if ep_corr_losses else float("nan")
    corr_ep_log.append(corr_ep)

    print(f"[Ep {ep+1:04d}] eps={eps:.3f} R={np.mean(ep_rewards):+.4f} "
          f"U={np.mean(ep_Us):.3f} ρ={np.mean(ep_rhos):.3f} "
          f"TD={td_ep:.5f} Corr={corr_ep:.5f}")

    if (ep + 1) % TARGET_UPDATE == 0:
        target_net.encoder.load_state_dict(main_net.encoder.state_dict())
        target_net.w.data.copy_(main_net.w.data)

    eps = max(eps_end, eps * eps_decay)

# Save encoder φ(s)
phi_path = DATA_DIR / f"{TOPO_NAME}_phi_trained_vanilla.pth"
torch.save(main_net.encoder.state_dict(), phi_path)
print(f"Saved vanilla encoder to {phi_path}")

# Save logs to CSV
df_summary = pd.DataFrame({
    "episode": np.arange(len(rewards_log)),
    "reward": rewards_log,
    "U": U_log,
    "rho": rho_log,
    "corr_loss": corr_ep_log
})
out_csv = f"{SAVE_DIR}/train_reward_utilization_corr_summary_vanilla-try.csv"
df_summary.to_csv(out_csv, index=False)
print(f"Saved: {out_csv}")


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import os

SAVE_DIR = "./plots"
os.makedirs(SAVE_DIR, exist_ok=True)

# Load both CSV logs
df_ds = pd.read_csv(f"{SAVE_DIR}/train_reward_utilization_corr_summary_dsdqn.csv")                 # DS-DQN
df_vanilla = pd.read_csv(f"{SAVE_DIR}/train_reward_utilization_corr_summary_vanilla-try.csv")    # Vanilla DQN

# Extract correlation logs
corr_ds = df_ds["corr_loss"].values
corr_vanilla = df_vanilla["corr_loss"].values

# Rolling mean helper
def roll(x, w=50):
    return pd.Series(x).rolling(window=w, min_periods=1).mean().to_numpy()

ROLL_WIN = 50

# ==========================================================
# Combined Correlation Plot: DS-DQN vs Vanilla DQN
# ==========================================================
plt.figure()

plt.plot(
    roll(corr_ds, ROLL_WIN),
    color="darkorange", linewidth=2.3,
    label="DS-DQN"
)
plt.plot(
    roll(corr_vanilla, ROLL_WIN),
    color="crimson", linewidth=2.3,
    label="Vanilla DQN"
)

# Titles and labels
plt.title("Mean Correlation Loss per Episode", fontsize=16, weight='semibold')
plt.xlabel("Episode", fontsize=15, weight='bold')
plt.ylabel("Mean correlation penalty", fontsize=15, weight='bold')

# Axis styling
plt.xticks(fontsize=13, weight='bold')
plt.yticks(fontsize=13, weight='bold')
plt.tick_params(axis='both', which='major', width=1.2, length=6)

# Grid and legend
plt.grid(True, linestyle="--", alpha=0.35)
plt.legend(fontsize=13, loc="upper right", frameon=False)

# Save high-quality EPS for Overleaf
plt.tight_layout()
plt.savefig(f"{SAVE_DIR}/train_corrloss_comparison-try.eps",
            format='eps', dpi=600, bbox_inches="tight")
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import os

SAVE_DIR = "./plots"
os.makedirs(SAVE_DIR, exist_ok=True)

# Load both CSV logs
df_ds = pd.read_csv(f"{SAVE_DIR}/train_reward_utilization_corr_summary_dsdqn.csv")                 # DS-DQN
df_vanilla = pd.read_csv(f"{SAVE_DIR}/train_reward_utilization_corr_summary_vanilla-try.csv")    # Vanilla DQN

# Extract correlation logs
corr_ds = df_ds["corr_loss"].values
corr_vanilla = df_vanilla["corr_loss"].values

# Rolling mean helper
def roll(x, w=50):
    return pd.Series(x).rolling(window=w, min_periods=1).mean().to_numpy()

ROLL_WIN = 50

# ==========================================================
# Combined Correlation Plot: DS-DQN vs Vanilla DQN
# ==========================================================
plt.figure()

# DS-DQN first
plt.plot(
    roll(corr_ds, ROLL_WIN),
    color="blue", linewidth=2.5,
    label="DS-DQN"
)
# Vanilla DQN next
plt.plot(
    roll(corr_vanilla, ROLL_WIN),
    color="red", linewidth=2.5,
    label="Vanilla DQN"
)

# Titles and labels
plt.title("Mean Correlation Loss per Episode", fontsize=16, weight='semibold')
plt.xlabel("Episode", fontsize=15, weight='bold')
plt.ylabel("Mean correlation penalty", fontsize=15, weight='bold')

# Axis styling
plt.xticks(fontsize=13, weight='bold')
plt.yticks(fontsize=13, weight='bold')
plt.tick_params(axis='both', which='major', width=1.2, length=6)
plt.ylim(0, 1.0)

# Grid
plt.grid(True, linestyle="--", alpha=0.35)

# Legend inside the plot, centered at the bottom, with box visible
plt.legend(
    fontsize=13,
    loc='center right',
    #bbox_to_anchor=(0.5, 0.05),   # inside the plot
    #ncol=1,
    frameon=True,
    framealpha=0.85,
    facecolor='white',
    edgecolor='black'
)

# Save high-quality EPS for Overleaf
plt.tight_layout()
#plt.savefig(f"{SAVE_DIR}/train_corrloss_comparison-try.png",
 #           format='eps', dpi=600, bbox_inches="tight")
plt.savefig("train_corrloss_comparison.png", dpi=300, bbox_inches="tight")
#plt.savefig("train_reward_utilization_overall_clean.png", dpi=300, bbox_inches="tight")


plt.show()


In [ ]:
# Cell 7 — Inference (Algorithm 2, training-consistent with τ filtering + ε-decay per step + α decay)

def run_inference_alg2(
    TMS_test, META_test, phi_encoder,
    lam=0.7, episodes=1000, K=30,
    alpha0=1e-3, alpha_min=1e-5, alpha_decay=0.999,
    gamma=0.99,
    eps_start=1.0, eps_end=0.05, eps_decay=0.995
):
    """
    Inference (Alg 2):
    - Encoder φ(s) frozen; only w updated online with row-wise LFA.
    - ε-greedy with decay (decayed *per step*).
    - α decays per episode: large at start, smaller as adaptation continues.
    - τ filtering for path selection.
    """
    phi_encoder.eval()
    with torch.no_grad():
        phi_dim = phi_encoder(torch.zeros(1, STATE_DIM).to(device)).size(1)

    # Linear FA weights for all actions (start near 0 for stability)
    w = torch.zeros(N_ACTIONS, phi_dim, device=device)

    rewards_log, U_log, rho_log, gen_log = [], [], [], []

    # ε-decay init
    eps = eps_start
    from tqdm import trange

    for ep in trange(episodes, desc="Inference Episodes", ncols=100):

   # for ep in range(episodes):
        # 🔑 scheduled learning rate for this episode
        alpha = max(alpha_min, alpha0 * (alpha_decay ** ep))

        # sample a TM from test set
        idx = np.random.randint(len(TMS_test))
        TM = TMS_test[idx]
        tm_type = META_test.iloc[idx]["gen"]
        state = init_state(TM)

        ep_rewards, ep_Us, ep_rhos = [], [], []
        ep_deltas = []
        explore_ct = 0; exploit_ct = 0

        for step in range(K):
            # φ(s)
            s_tensor = torch.FloatTensor(state).unsqueeze(0).to(device)
            with torch.no_grad():
                phi_s = phi_encoder(s_tensor).squeeze(0)     # [d]

            # Q(s,·) = φ(s) @ w^T
            q_vals = torch.mv(w, phi_s)                      # [A]

            # ε-greedy with decaying ε
            if np.random.rand() < eps:
                action = np.random.randint(N_ACTIONS); explore_ct += 1
            else:
                action = int(torch.argmax(q_vals).item()); exploit_ct += 1

            # env step (τ filtering consistent with training)
            s_next, r, U, rho = apply_action(state, action, TM, lam=lam, tau=0.8)

            # TD target
            with torch.no_grad():
                phi_sp = phi_encoder(torch.FloatTensor(s_next).unsqueeze(0).to(device)).squeeze(0)
                q_next = torch.mv(w, phi_sp)
                y = torch.tensor(r, device=device) + gamma * torch.max(q_next)

            # δ = y - Q(s,a)
            q_sa = torch.dot(w[action], phi_s)
            delta = (y - q_sa).detach()
            ep_deltas.append(float(delta))

            # Row-wise update with decayed α
            with torch.no_grad():
                w[action].add_(alpha * delta * phi_s)

            # advance
            state = s_next
            ep_rewards.append(r); ep_Us.append(U); ep_rhos.append(rho)

            # ✅ decay ε per step
            eps = max(eps_end, eps * eps_decay)

        # episode logs
        Rm = float(np.mean(ep_rewards)); Um = float(np.mean(ep_Us)); rhom = float(np.mean(ep_rhos))
        rewards_log.append(Rm); U_log.append(Um); rho_log.append(rhom); gen_log.append(tm_type)

        # safe average for δ̄
        if not ep_deltas:
            d_mean = float('nan')
        else:
            d_mean = float(np.mean(ep_deltas))
    
        print(
            f"[Alg2 Ep {ep+1:03d}] λ={lam:.2f} eps={eps:.3f} α={alpha:.6f} "
            f"R={Rm:+.4f} U={Um:.3f} ρ={rhom:.3f} "
            f"δ̄={d_mean:+.5f} stepsE={explore_ct} stepsX={exploit_ct} tm={tm_type}"
        )

    return rewards_log, U_log, rho_log, gen_log


In [ ]:
class IdentityEncoder(nn.Module):
    def __init__(self, state_dim):
        super().__init__()
        self.state_dim = state_dim

    def forward(self, x):
        # Directly return raw state (no normalization)
        return x


In [ ]:
class RandomEncoder(nn.Module):
    def __init__(self, state_dim, phi_dim=None):
        super().__init__()
        if phi_dim is None:
            phi_dim = state_dim   # match transfer and scratch → same dim

        self.proj = nn.Linear(state_dim, phi_dim, bias=False)
        # freeze weights (no training)
        with torch.no_grad():
            self.proj.weight.copy_(torch.randn(phi_dim, state_dim) * 0.01)
        for p in self.parameters():
            p.requires_grad = False

    def forward(self, x):
        # Random projection without normalization
        return self.proj(x)


In [ ]:
# Cell 8 — Run inference: Transfer vs Scratch (with α schedule)

# 1) Load trained φ(s) and freeze
phi_trained = PhiEncoder(STATE_DIM, phi_dim=None).to(device)  # → 293
phi_trained.load_state_dict(torch.load(DATA_DIR / f"{TOPO_NAME}_phi_trained_DSDQN-1.pth",
                                       map_location=device))
phi_trained.eval()
for p in phi_trained.parameters():
    p.requires_grad = False

# 2) Scratch φ(s): identity mapping (raw state vector as φ(s))
phi_scratch = IdentityEncoder(STATE_DIM).to(device)
phi_scratch.eval()

# 3) Random φ(s): fixed random projection (now 293-dim too)
phi_random = RandomEncoder(STATE_DIM, phi_dim=None).to(device)
phi_random.eval()

# Common inference settings
LAM_TEST   = 0.55      # different operating point than training → good for transfer
EPISODES_I = 1000
K_I        = 30
GAMMA_I    = 0.99


N_RUNS = 10
transfer_times = []

for run in range(1, N_RUNS + 1):
    print(f"Starting Transfer Run {run}/{N_RUNS}...")
    start_inf = time.perf_counter()

    _ = run_inference_alg2(
        TMS_test, META_test, phi_trained,
        lam=LAM_TEST, episodes=EPISODES_I, K=K_I,
        alpha0=ALPHA0_T, alpha_min=ALPHA_MIN_T, alpha_decay=DECAY_T,
        gamma=GAMMA_I,
        eps_start=1.0, eps_end=0.05, eps_decay=0.995
    )

    end_inf = time.perf_counter()
    transfer_times.append(end_inf - start_inf)
    print(f"Run {run}: {transfer_times[-1]:.2f} seconds")

print("\n=== Transfer Inference Timing Summary (10 Runs) ===")
print(f"Average Time: {np.mean(transfer_times):.2f} s ± {np.std(transfer_times):.2f} s")


In [ ]:
# =======================================
# ⚡ Clean Multi-run Inference Summary (no timing info)
# =======================================
import numpy as np, torch, pandas as pd, os

# --- Configurable settings ---
N_RUNS = 10
LAM_TEST = 0.55
EPISODES_I = 1000
K_I = 30
GAMMA_I = 0.99

R_THRESH = 0.6
N_CONSEC = 30
BETA = 0.05
WINDOW = 100
SAVE_DIR = "./plots"
os.makedirs(SAVE_DIR, exist_ok=True)

# α schedules
ALPHA0_T, ALPHA_MIN_T, DECAY_T = 1e-5, 3.5e-6, 0.999
ALPHA0_S, ALPHA_MIN_S, DECAY_S = 3e-4, 1e-5, 0.999

# --- Load encoders once ---
phi_trained = PhiEncoder(STATE_DIM, phi_dim=None).to(device)
phi_trained.load_state_dict(torch.load(DATA_DIR / f"{TOPO_NAME}_phi_trained.pth", map_location=device))
phi_trained.eval()
for p in phi_trained.parameters(): p.requires_grad = False

phi_scratch = IdentityEncoder(STATE_DIM).to(device); phi_scratch.eval()
phi_random  = RandomEncoder(STATE_DIM, phi_dim=None).to(device); phi_random.eval()

# --- Helper functions ---
def ema(series, beta=BETA):
    s = series[0]; out = []
    for x in series:
        s = beta*s + (1-beta)*x
        out.append(s)
    return np.array(out)

def find_first_above_threshold(rewards, thresh, consec=1):
    streak = 0
    for i, r in enumerate(rewards, start=1):
        if r >= thresh:
            streak += 1
            if streak >= consec: return i - consec + 1
        else:
            streak = 0
    return None

def last_ema_stats(rewards, U, rho, beta=BETA, window=WINDOW):
    r, u, rh = ema(rewards, beta)[-window:], ema(U, beta)[-window:], ema(rho, beta)[-window:]
    return np.mean(r), np.mean(u), np.mean(rh)

def compute_run_summary(rewards_T, U_T, rho_T, rewards_S, U_S, rho_S, rewards_R, U_R, rho_R):
    return {
        "Transfer": (find_first_above_threshold(rewards_T,R_THRESH,N_CONSEC), *last_ema_stats(rewards_T,U_T,rho_T)),
        "Scratch" : (find_first_above_threshold(rewards_S,R_THRESH,N_CONSEC), *last_ema_stats(rewards_S,U_S,rho_S)),
        "Random"  : (find_first_above_threshold(rewards_R,R_THRESH,N_CONSEC), *last_ema_stats(rewards_R,U_R,rho_R))
    }

# --- Storage ---
all_runs = {"Transfer": [], "Scratch": [], "Random": []}

# --- Run loop ---
for run_id in range(N_RUNS):
    print(f"\n========== RUN {run_id+1}/{N_RUNS} ==========")

    print("  → Transfer inference...")
    rewards_T, U_T, rho_T, _ = run_inference_alg2(
        TMS_test, META_test, phi_trained, lam=LAM_TEST,
        episodes=EPISODES_I, K=K_I,
        alpha0=ALPHA0_T, alpha_min=ALPHA_MIN_T, alpha_decay=DECAY_T,
        gamma=GAMMA_I, eps_start=1.0, eps_end=0.05, eps_decay=0.995
    )

    print("  → Scratch inference...")
    rewards_S, U_S, rho_S, _ = run_inference_alg2(
        TMS_test, META_test, phi_scratch, lam=LAM_TEST,
        episodes=EPISODES_I, K=K_I,
        alpha0=ALPHA0_S, alpha_min=ALPHA_MIN_S, alpha_decay=DECAY_S,
        gamma=GAMMA_I, eps_start=1.0, eps_end=0.05, eps_decay=0.995
    )

    print("  → Random inference...")
    rewards_R, U_R, rho_R, _ = run_inference_alg2(
        TMS_test, META_test, phi_random, lam=LAM_TEST,
        episodes=EPISODES_I, K=K_I,
        alpha0=ALPHA0_S, alpha_min=ALPHA_MIN_S, alpha_decay=DECAY_S,
        gamma=GAMMA_I, eps_start=1.0, eps_end=0.05, eps_decay=0.995
    )

    res = compute_run_summary(rewards_T, U_T, rho_T,
                              rewards_S, U_S, rho_S,
                              rewards_R, U_R, rho_R)
    for mode in all_runs:
        all_runs[mode].append(res[mode])

    print("  ✓ Transfer / Scratch / Random completed.")

# --- Aggregate ---
def summarize_mode(data):
    arr = np.array(data, dtype=float)
    return arr.mean(axis=0), arr.std(axis=0)

rows = []
print("\n================ FINAL SUMMARY (Mean ± Std over runs) ================")
for mode, data in all_runs.items():
    mean, std = summarize_mode(data)
    print(f"\n{mode}\n  Earliest_Ep  : {mean[0]:.1f} ± {std[0]:.1f}\n  Reward       : {mean[1]:.3f} ± {std[1]:.3f}\n  Max Util     : {mean[2]:.3f} ± {std[2]:.3f}\n  Avg Util     : {mean[3]:.3f} ± {std[3]:.3f}")
    rows.append({
        "mode": mode,
        "Earliest_Ep (≥0.6×30)": f"{mean[0]:.1f} ± {std[0]:.1f}",
        "Reward": f"{mean[1]:.3f} ± {std[1]:.3f}",
        "Max Util": f"{mean[2]:.3f} ± {std[2]:.3f}",
        "Avg Util": f"{mean[3]:.3f} ± {std[3]:.3f}"
    })

# --- Save results ---
df_summary = pd.DataFrame(rows)
df_summary.to_csv(f"{SAVE_DIR}/multi_run_summary_0.55_10runs-node30_copy4.csv", index=False)
print(f"\nSaved summary → {SAVE_DIR}/multi_run_summary_0.55_10runs-node30_copy4.csv")
display(df_summary)


In [ ]:
# =======================================
# ⚡ Vanilla DQN Inference (10 runs)
# =======================================
import numpy as np, torch, pandas as pd, os

# --- Configurable settings ---
N_RUNS = 10
LAM_TEST = 0.55
EPISODES_I = 1000
K_I = 30
GAMMA_I = 0.99

R_THRESH = 0.6
N_CONSEC = 30
BETA = 0.05
WINDOW = 100
SAVE_DIR = "./plots"
os.makedirs(SAVE_DIR, exist_ok=True)

# α schedules
ALPHA0_T, ALPHA_MIN_T, DECAY_T = 1e-5, 3.5e-6, 0.999
# ALPHA0_S, ALPHA_MIN_S, DECAY_S = 3e-4, 1e-5, 0.999
# --- Load vanilla encoder ---
phi_vanilla = PhiEncoder(STATE_DIM, phi_dim=None).to(device)
phi_vanilla.load_state_dict(
    torch.load(DATA_DIR / f"{TOPO_NAME}_phi_trained_vanilla.pth", map_location=device)
)
phi_vanilla.eval()
for p in phi_vanilla.parameters():
    p.requires_grad = False

# --- Helper functions ---
def ema(series, beta=BETA):
    s = series[0]; out = []
    for x in series:
        s = beta*s + (1-beta)*x
        out.append(s)
    return np.array(out)

def find_first_above_threshold(rewards, thresh, consec=1):
    streak = 0
    for i, r in enumerate(rewards, start=1):
        if r >= thresh:
            streak += 1
            if streak >= consec: return i - consec + 1
        else:
            streak = 0
    return None

def last_ema_stats(rewards, U, rho, beta=BETA, window=WINDOW):
    r, u, rh = ema(rewards, beta)[-window:], ema(U, beta)[-window:], ema(rho, beta)[-window:]
    return np.mean(r), np.mean(u), np.mean(rh)

# --- Storage ---
all_runs_vanilla = []

for run_id in range(N_RUNS):
    print(f"\n========== VANILLA RUN {run_id+1}/{N_RUNS} ==========")

    rewards_V, U_V, rho_V, _ = run_inference_alg2(
        TMS_test, META_test, phi_vanilla, lam=LAM_TEST,
        episodes=EPISODES_I, K=K_I,
        alpha0=ALPHA0_T, alpha_min=ALPHA_MIN_T, alpha_decay=DECAY_T,
        gamma=GAMMA_I, eps_start=1.0, eps_end=0.05, eps_decay=0.995
    )

    # --- Compute metrics ---
    earliest_ep = find_first_above_threshold(rewards_V, R_THRESH, N_CONSEC)
    if earliest_ep is None:
        earliest_ep = EPISODES_I   # handle no convergence

    r_mean, u_mean, rho_mean = last_ema_stats(rewards_V, U_V, rho_V)

    all_runs_vanilla.append([earliest_ep, r_mean, u_mean, rho_mean])


# --- Convert to array ---
arr = np.array(all_runs_vanilla, dtype=float)

# --- Mean & Std ---
mean = arr.mean(axis=0)
std  = arr.std(axis=0)

print("\n================ VANILLA SUMMARY (Mean ± Std over runs) ================")
print(f"Earliest_Ep  : {mean[0]:.1f} ± {std[0]:.1f}")
print(f"Reward       : {mean[1]:.3f} ± {std[1]:.3f}")
print(f"Max Util     : {mean[2]:.3f} ± {std[2]:.3f}")
print(f"Avg Util     : {mean[3]:.3f} ± {std[3]:.3f}")


# --- Save ---
df_vanilla = pd.DataFrame([{
    "mode": "Vanilla",
    "Earliest_Ep (≥0.6×30)": f"{mean[0]:.1f} ± {std[0]:.1f}",
    "Reward": f"{mean[1]:.3f} ± {std[1]:.3f}",
    "Max Util": f"{mean[2]:.3f} ± {std[2]:.3f}",
    "Avg Util": f"{mean[3]:.3f} ± {std[3]:.3f}"
}])

out_path = f"{SAVE_DIR}/vanilla_summary.csv"
df_vanilla.to_csv(out_path, index=False)

print(f"\nSaved vanilla summary → {out_path}")
display(df_vanilla)